# AIシフト作成ツール

既存のシフト表Excel（.xlsm）の「個人シフト入力表」をAIが自動で埋めます。
上のセルから順に ▶ を押してください。

**はじめて使うとき**: 完成済みの月のExcelをアップロードし、モードを「学習」にして実行します。
スタッフごとの「使うシフト・働く曜日・出勤日数」を学習した `AI設定.xlsx` ができるので、ダウンロードして必要なら直してください。

**毎月の作成**: 新しい月のExcelと `AI設定.xlsx` をアップロードし、モードを「作成」にして実行します。

「個人シフト入力表」に入っている値は希望として扱います。

| 入力 | 意味 |
|---|---|
| シフトNO. | その日はそのシフトで確定 |
| 0 | 休み希望（確定） |
| 空欄 | AIが決める |

In [ ]:
#@title ① 準備（最初に1回）
!pip -q install ortools openpyxl
import os, urllib.request
for branch in ["main", "claude/sleepy-wozniak-ucx993"]:
    try:
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/ngsw6126/Cash-Register/{branch}/shift-ai/shift_ai.py", "shift_ai.py")
        break
    except Exception:
        pass
import importlib, shift_ai
importlib.reload(shift_ai)
print("準備完了")

In [ ]:
#@title ② ファイルをアップロード（Excel と、あれば AI設定.xlsx）
from google.colab import files
uploaded = files.upload()
excel = next((f for f in uploaded if f.lower().endswith((".xlsm", ".xlsx")) and "AI設定" not in f), None)
settings = next((f for f in uploaded if "AI設定" in f), "AI設定.xlsx" if os.path.exists("AI設定.xlsx") else None)
print("Excel:", excel)
print("設定:", settings or "なし（Excelの内容から学習します）")

In [ ]:
#@title ③ 実行
モード = "作成"  #@param ["作成", "学習"]
入力済みのシフトを消して全部作り直す = False  #@param {type:"boolean"}
計算時間_秒 = 60  #@param {type:"integer"}

if モード == "学習":
    shift_ai.main(["learn", excel, "-s", "AI設定.xlsx"])
    files.download("AI設定.xlsx")
else:
    args = ["generate", excel, "-o", "AIシフト完成_" + os.path.basename(excel),
            "-r", "AIシフト_レポート.xlsx", "-t", str(計算時間_秒)]
    if settings:
        args += ["-s", settings]
    if 入力済みのシフトを消して全部作り直す:
        args.append("--clear")
    out = shift_ai.main(args)
    files.download(out)
    files.download("AIシフト_レポート.xlsx")